# pt_createverdicts

Loads today's race JSONs (from PT_Vorarbeiten precomputed pickle) and saves
per-meeting verdict JSONs to Google Drive.

**Requires:** `precomputed_tdy_{TODAY}.pkl` produced by PT_Vorarbeiten.

**Produces:**
- `race_verdicts/{date}_{meeting}.json` — race JSONs with runners + ML predictions

In [ ]:
import os, json, pickle
from datetime import date

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE  = '/content/drive/MyDrive/PT' if os.path.exists('/content/drive/MyDrive/PT') else '.'
TODAY = date.today().strftime('%Y-%m-%d')
print(f'BASE:  {BASE}')
print(f'TODAY: {TODAY}')

In [ ]:
# ── Load precomputed from PT_Vorarbeiten ────────────────────────────────────
_pkl_path = f'{BASE}/precomputed_tdy_{TODAY}.pkl'
with open(_pkl_path, 'rb') as _f:
    _precomputed = pickle.load(_f)

race_jsons = _precomputed['race_jsons']
print(f'✅ Loaded precomputed_tdy_{TODAY}.pkl  (race_jsons: {len(race_jsons)})')

In [ ]:
# ── Save race JSONs per meeting to PT/race_verdicts/ ────────────────────────
_rv_dir = f'{BASE}/race_verdicts'
os.makedirs(_rv_dir, exist_ok=True)

_by_meeting = {}
for race_key, race_json in race_jsons.items():
    _meeting = race_json.get('meeting', 'unknown')
    _safe_m  = _meeting.replace(' ', '_').replace('/', '_')
    if _safe_m not in _by_meeting:
        _by_meeting[_safe_m] = []

    _by_meeting[_safe_m].append({
        'raceId':           race_json.get('raceId'),
        'race':             race_json.get('race'),
        'meeting':          _meeting,
        'going':            race_json.get('going'),
        'going_category':   race_json.get('going_category'),
        'distance_m':       race_json.get('distance_m'),
        'distance_group':   race_json.get('distance_group'),
        'race_type':        race_json.get('race_type'),
        'race_class':       race_json.get('race_class'),
        'total_prize_eur':  race_json.get('total_prize_eur'),
        'field_size':       race_json.get('field_size'),
        'paristurf_verdict': race_json.get('paristurf_verdict'),
        'horses':           race_json.get('horses', []),
    })

for _safe_m, _races in _by_meeting.items():
    _jpath = f'{_rv_dir}/{TODAY}_{_safe_m}.json'
    with open(_jpath, 'w', encoding='utf-8') as _f:
        json.dump(_races, _f, ensure_ascii=False, indent=2, default=str)
    print(f'  Saved {_jpath} ({len(_races)} races)')

In [ ]:
print('\n✅ pt_createverdicts complete.')
print(f'   Races processed:  {len(race_jsons)}')
print(f'   race_verdicts/: {TODAY}_*.json')
print('\nNow run PT_Create_HTMLs_fast.ipynb ▶')